# Feature Extraction v2 — YAMNet mean+std (D1 & D2)

Varian dari `04_features_yamnet.ipynb`. **Satu-satunya perubahan:** cara meringkas frame.

Baseline meringkas ~6 frame YAMNet jadi **rata-rata** saja (1024-d) — itu membuang variasi
antar-waktu. Pembeda sirine justru pola modulasi temporal (wailing naik-turun). Di sini kita
gabung **mean + std** antar-frame → **2048-d**, agar variasi temporal ikut terekam.

- Cache baru `ml/cache/yamnet_stats_{d1,d2}.npz` — **tidak menimpa** cache lama.
- Semua lain identik: 16 kHz, mono, peak-norm, 3.0 s; `TF_USE_LEGACY_KERAS=1`.

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import time
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ML_DIR = ROOT / "ml"
CACHE_DIR = ML_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

YAMNET_SR = 16000
DURATION = 3.0
N_SAMPLES = int(YAMNET_SR * DURATION)

DATASETS = {"d1": ML_DIR / "manifest_d1.csv", "d2": ML_DIR / "manifest_d2.csv"}

print(f"tensorflow {tf.__version__} · keras legacy = {os.environ['TF_USE_LEGACY_KERAS']}")
for name, m in DATASETS.items():
    assert m.exists(), f"manifest {name} belum ada: {m}"
print("manifest D1 & D2 ditemukan.")

tensorflow 2.16.2 · keras legacy = 1
manifest D1 & D2 ditemukan.


## 1 · Muat YAMNet & fungsi ekstraksi

In [2]:
t0 = time.time()
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
print(f"YAMNet dimuat ({time.time() - t0:.1f}s)")


def load_16k(path: str) -> np.ndarray:
    """16 kHz, mono, peak-normalized, panjang tetap N_SAMPLES."""
    y, _ = librosa.load(path, sr=YAMNET_SR, mono=True)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    y = y[:N_SAMPLES].astype(np.float32)
    peak = np.abs(y).max()
    return y / peak if peak > 0 else y


def embed(path: str) -> np.ndarray:
    """concat(mean, std) embedding antar-frame -> 2048-d."""
    _scores, emb, _spec = yamnet(load_16k(path))
    e = emb.numpy()                                    # (n_frames, 1024)
    return np.concatenate([e.mean(axis=0), e.std(axis=0)]).astype(np.float32)


# validasi bentuk
sample = ROOT / pd.read_csv(DATASETS["d1"]).iloc[0].path
v = embed(str(sample))
EMB_DIM = int(v.shape[0])
assert EMB_DIM == 2048, f"harusnya 2048, dapat {EMB_DIM}"
print(f"validasi ok — embedding {v.shape} (mean 1024 + std 1024) dari {sample.name}")

YAMNet dimuat (5.2s)


validasi ok — embedding (2048,) (mean 1024 + std 1024) dari sound_1.wav


## 2 · Ekstrak & cache kedua dataset

In [3]:
def extract_dataset(name: str, manifest_path: Path) -> dict:
    df = pd.read_csv(manifest_path)
    n = len(df)
    embs = np.zeros((n, EMB_DIM), dtype=np.float32)

    print(f"[{name}] {n} file ...")
    t0 = time.time()
    for i, row in enumerate(df.itertuples(index=False)):
        embs[i] = embed(str(ROOT / row.path))
        if (i + 1) % 250 == 0 or i + 1 == n:
            print(f"  {i + 1:>4}/{n}  ({(i + 1) / (time.time() - t0):.0f} file/s)")

    out = CACHE_DIR / f"yamnet_stats_{name}.npz"
    np.savez(out, emb=embs, filename=df.filename.to_numpy(),
             label=df.label.to_numpy(), source_id=df.source_id.to_numpy())
    print(f"[{name}] tersimpan -> {out}  ({embs.shape})  {time.time() - t0:.0f}s\n")
    return {"name": name, "shape": embs.shape}


results = [extract_dataset(name, path) for name, path in DATASETS.items()]

[d1] 596 file ...


   250/596  (34 file/s)


   500/596  (34 file/s)


   596/596  (34 file/s)
[d1] tersimpan -> D:\Coding Vscode\Siren Classification\ml\cache\yamnet_stats_d1.npz  ((596, 2048))  17s

[d2] 1675 file ...


   250/1675  (34 file/s)


...

   750/1675  (34 file/s)


  1000/1675  (34 file/s)


  1250/1675  (34 file/s)


  1500/1675  (34 file/s)


  1675/1675  (34 file/s)
[d2] tersimpan -> D:\Coding Vscode\Siren Classification\ml\cache\yamnet_stats_d2.npz  ((1675, 2048))  49s



## 3 · Verifikasi cache

In [4]:
for name in DATASETS:
    data = np.load(CACHE_DIR / f"yamnet_stats_{name}.npz", allow_pickle=True)
    emb, fnames = data["emb"], data["filename"]
    print(f"[{name}] emb {emb.shape} · {len(fnames)} filename · NaN={np.isnan(emb).any()}")
    cached = set(fnames)
    for split in ("train", "val", "test"):
        sp = pd.read_csv(ML_DIR / f"split_{name}_{split}.csv")
        missing = set(sp.filename) - cached
        assert not missing, f"{name}/{split}: {len(missing)} file tak ada di cache!"
        print(f"    {split:5s} {len(sp):>4} file -> semua ada di cache")
print("\nOK — cache mean+std lengkap & konsisten dengan split.")

[d1] emb (596, 2048) · 596 filename · NaN=False
    train  426 file -> semua ada di cache
    val     85 file -> semua ada di cache
    test    85 file -> semua ada di cache
[d2] emb (1675, 2048) · 1675 filename · NaN=False
    train 1195 file -> semua ada di cache
    val    240 file -> semua ada di cache
    test   240 file -> semua ada di cache

OK — cache mean+std lengkap & konsisten dengan split.


---

**Selanjutnya:** `08_experiment_meanstd.ipynb` melatih head yang sama (recipe esf1) di atas
fitur 2048-d ini, lalu membandingkan macro-F1 dengan baseline & esf1.